# KOSPI200 옵션 VIX6 Case 1 재현 노트북

KOSPI200 옵션을 Cboe식 여섯 성분으로 근사한 뒤 기존 Robust VKOSPI, VIX6 혼합형, 로지스틱 입력형, VIX6 단독형을 같은 4자산·비용 조건에서 비교합니다. 옵션은 신호 계산에만 쓰며 투자자산으로 편입하지 않습니다.

> 현재 연구 결론: 2018~2026년 CAGR·Sharpe·MDD를 동시에 개선한 VIX6 대안이 없어 기존 최종 전략을 유지합니다.

## 1. 프로젝트 준비

Colab의 `/content/RegimeDecisionTest`에 프로젝트가 없으면 ZIP 파일을 선택합니다. ZIP 안에는 최소한 `raw_data/`, `cache/`, `results/openassetpricing_composites.csv`와 프로젝트의 파이썬 파일이 있어야 합니다. 860,380행 옵션 CSV가 크므로 Google Drive에 프로젝트 폴더를 두고 `PROJECT_DIR`만 바꿔도 됩니다.

In [ ]:
from pathlib import Path
import os, sys, zipfile

PROJECT_DIR = Path('/content/RegimeDecisionTest')

if not PROJECT_DIR.exists():
    from google.colab import files
    print('RegimeDecisionTest 프로젝트 ZIP을 선택하세요.')
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall('/content')

assert PROJECT_DIR.exists(), f'프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('작업 폴더:', PROJECT_DIR)

## 2. 의존성 설치와 입력 파일 점검

In [ ]:
%pip -q install pandas numpy scipy scikit-learn openpyxl matplotlib

required = [
    PROJECT_DIR / 'raw_data' / 'KOSPI200OptionPrice.csv',
    PROJECT_DIR / 'raw_data' / 'VKOSPIData.csv',
    PROJECT_DIR / 'cache' / 'market_daily.csv',
    PROJECT_DIR / 'results' / 'openassetpricing_composites.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, '필수 파일 누락:\n' + '\n'.join(missing)
print('필수 입력 파일 확인 완료')

## 3. VIX6 일별 분해값 생성

기존 캐시가 있으면 바로 읽습니다. 옵션 원본부터 다시 계산하려면 `FORCE_FEATURES=True`로 바꾸세요. 원본 재계산은 Colab에서 시간이 꽤 걸릴 수 있습니다.

In [ ]:
from vix6_case1_strategy import build_vix6_features

FORCE_FEATURES = False
features = build_vix6_features(force=FORCE_FEATURES)
display(features.tail())
print(f'{len(features):,}일 | {features.index.min().date()} ~ {features.index.max().date()}')
print('분해 항등식 최대 잔차:', features['decomposition_residual'].dropna().abs().max())

## 4. 네 전략 비교와 최종 선택

입력변수 후보 28개의 캐시가 있으면 최선 후보만 재현합니다. 후보 탐색 전체를 다시 돌리려면 `FORCE_SEARCH=True`로 바꾸세요.

In [ ]:
from vix6_case1_model_comparison import run_comparison

FORCE_SEARCH = False
report = run_comparison(force_search=FORCE_SEARCH)
print('최종 선택:', report['selected_strategy'])
print(report['decision'])

## 5. 성과표와 누적수익 확인

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.read_csv(PROJECT_DIR / 'results' / 'vix6_case1_final_model_comparison.csv')
locked = comparison.loc[comparison['Period'].eq('locked_2018_2026'),
                        ['Strategy', 'CAGR', 'Sharpe', 'MDD', 'AvgTurnover']].copy()
for column in ['CAGR', 'MDD', 'AvgTurnover']:
    locked[column] = (100 * locked[column]).map(lambda value: f'{value:.2f}%')
locked['Sharpe'] = locked['Sharpe'].map(lambda value: f'{value:.3f}')
display(locked)

paths = {
    'Existing': 'balanced_logistic_no_sjm_final_reconciled.csv',
    'VIX6 Hybrid': 'vix6_case1_reconciled.csv',
    'VIX6 Logistic Input': 'vix6_case1_input_best_reconciled.csv',
    'VIX6 Standalone': 'vix6_case1_standalone_reconciled.csv',
}
plt.figure(figsize=(12, 6))
for label, filename in paths.items():
    frame = pd.read_csv(PROJECT_DIR / 'results' / filename, index_col=0)
    frame.index = pd.PeriodIndex(frame.index, freq='M').to_timestamp()
    wealth = (1 + frame['return']).cumprod()
    plt.plot(wealth.index, wealth, label=label, linewidth=2 if label == 'Existing' else 1.2)
plt.yscale('log')
plt.title('Four-asset strategy wealth paths (log scale)')
plt.grid(alpha=.25)
plt.legend()
plt.show()

## 6. 입력변수와 시차 감사

In [ ]:
print('기본 입력변수')
for name in report['input_candidate']['base_features']:
    print(' -', name)
print('\n추가한 VIX6 가공값')
for name in report['input_candidate']['added_vix6_features']:
    print(' -', name)
print('\n룩어헤드 감사')
for key, value in report['lookahead_audit'].items():
    print(f' - {key}: {value}')

## 해석 주의

KOSPI200 옵션 파일에는 bid·ask가 없어 이 노트북의 VIX6 값은 공식 VIX 포인트 기여도가 아니라 종가 IV 프록시입니다. 과거 성과는 미래 수익을 보장하지 않으며, 이번 비교는 잠금 구간을 개발 중 확인한 사후 탐색입니다.